# Step 5 — Calibration & Backtest

Consumes the Step 2 (`tier1_composite.csv`) and Step 4 (`step4_trigger_episodes.csv`,
`step4_final_risk_scores.json`) outputs and produces the concrete numbers plan Step 5/6 call
for, without inventing labels you haven't actually assigned:

1. **Backtest candidate mining** — auto-detects two kinds of candidate events (Brent price
   shocks, corridor news spikes) and exports a CSV template with the label columns *blank* for
   you to hand-fill (`is_genuine_event`, `expected_severity`, `notes`) per plan Step 5.1. This
   notebook does not fabricate ground truth.
2. **Lead-time metric** (Step 5.3 / Step 6) — computed directly against price-shock candidates,
   since a large Brent move is itself an objective event that doesn't need a manual label.
3. **Trigger-threshold sweep** (Step 5.4) — the Step 4 notebook flagged that 33.1% of
   corridor-timestamps were triggering, almost entirely on sub-score disagreement, and called
   that out as "first thing to tune in Step 5." This section gives you the actual rate table
   across a threshold grid instead of guessing at one new value.
4. **Human-review threshold** (Step 5.5) — data-driven percentile suggestion from the real
   `disagreement_index` distribution in your Step 4 output, instead of the flat placeholder
   0.30 currently in the Step 4 config.
5. **Injection-resistance check** (Step 6, retrieval-layer half) — injects one fabricated
   low-credibility "Hormuz closed" story into a copy of the RAG chunk store and confirms the
   credibility-tier-sort keeps it out of the top of the ranking. The Dove-suppression half of
   this test needs a live Hawk/Dove run and isn't repeated here to avoid duplicate LLM cost —
   see the note in Section 6.

**Out of scope here:** actually editing Step 4's config with the new thresholds (that's a
one-line change you make after reading the sweep table) and the manual labeling itself.

## 0. Config — check paths before running

In [1]:
import json
from pathlib import Path
from datetime import timedelta, datetime, timezone

import numpy as np
import pandas as pd
from scipy import sparse
import pickle

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 140)

# --- INPUTS ---
TIER1_COMPOSITE_PATH = Path(r"D:\python time\ET\code\risk_agent\code\outputs\tier1_composite.csv")
FUNDAMENTALS_PATH = Path(r"D:\python time\ET\code\risk_agent\csv_backfill\fundamentals\fundamentals_full.csv")
STEP4_EPISODES_PATH = Path("outputs/step4_trigger_episodes.csv")
STEP4_FINAL_RESULTS_PATH = Path("outputs/step4_final_risk_scores.json")
RAG_CHUNKS_PATH = Path("outputs/rag_chunks_metadata.csv")
RAG_MATRIX_PATH = Path("outputs/rag_tfidf_matrix.npz")
RAG_VECTORIZER_PATH = Path("outputs/rag_tfidf_vectorizer.pkl")

# --- OUTPUT ---
OUTPUT_DIR = Path("outputs")
BACKTEST_CANDIDATES_OUTPUT = OUTPUT_DIR / "step5_backtest_candidates.csv"
LEAD_TIME_OUTPUT = OUTPUT_DIR / "step5_lead_time_report.csv"
THRESHOLD_SWEEP_OUTPUT = OUTPUT_DIR / "step5_threshold_sweep.csv"
CALIBRATED_CONFIG_OUTPUT = OUTPUT_DIR / "step5_calibrated_config.json"

CORRIDORS = ["hormuz", "bab_el_mandeb", "suez", "red_sea", "persian_gulf", "india"]

# --- CANDIDATE MINING ---
PRICE_SHOCK_MOM_THRESHOLD = 0.08     # |month-over-month Brent % change| that counts as a "shock"
NEWS_SPIKE_PERCENTILE = 0.95         # per-corridor percentile on news_subscore that counts as a "spike"
NEWS_SPIKE_DEDUP_DAYS = 7            # collapse spikes within this many days per corridor to one candidate

# --- LEAD-TIME ---
LEAD_TIME_LOOKBACK_DAYS = 30         # how far back to look for a composite crossing before a price shock
COMPOSITE_TRIGGER_THRESHOLD_FOR_LEAD = 0.65   # same value Step 4 currently uses

# --- THRESHOLD SWEEP (Step 4's two trigger knobs) ---
COMPOSITE_THRESHOLD_GRID = [0.55, 0.60, 0.65, 0.70, 0.75]
DISAGREEMENT_THRESHOLD_GRID = [0.30, 0.35, 0.40, 0.45, 0.50]
CURRENT_COMPOSITE_THRESHOLD = 0.65   # Step 4's current config, for reference in the printout
CURRENT_DISAGREEMENT_THRESHOLD = 0.35

# --- HUMAN REVIEW THRESHOLD ---
HUMAN_REVIEW_PERCENTILE = 0.80       # suggest flagging the top 20% most-disagreed episodes
CURRENT_HUMAN_REVIEW_THRESHOLD = 0.30   # Step 4's current flat value, for reference

# --- INJECTION TEST ---
INJECTION_TEST_CORRIDOR = "hormuz"
INJECTION_TEST_QUERY = "strait of hormuz closed shipping halted tanker traffic suspended"
INJECTION_TEST_TOP_K = 8
INJECTION_TEST_LOOKBACK_DAYS = 14

## 1. Load Tier 1 composite history and Brent fundamentals

In [2]:
composite = pd.read_csv(TIER1_COMPOSITE_PATH)
composite["as_of"] = pd.to_datetime(composite["as_of"], utc=True)
print(f"Tier 1 composite: {len(composite):,} rows, corridors: {sorted(composite['corridor_id'].unique())}")

fund = pd.read_csv(FUNDAMENTALS_PATH)
fund["as_of"] = pd.to_datetime(fund["as_of"], utc=True)
brent = (
    fund[(fund["corridor_id"] == "global") & (fund["metric"] == "brent_price")]
    .sort_values("as_of")
    .set_index("as_of")["value"]
)
print(f"Brent series: {len(brent)} monthly points, {brent.index.min().date()} -> {brent.index.max().date()}")

Tier 1 composite: 9,174 rows, corridors: ['bab_el_mandeb', 'hormuz', 'india', 'persian_gulf', 'red_sea', 'suez']
Brent series: 12 monthly points, 2025-07-31 -> 2026-06-30


## 2. Mine backtest candidates (Step 5.1 scaffold)

Two independent detectors, merged into one labeling template. Neither assigns a label — that
column stays blank for you.

In [3]:
# --- Price-shock candidates: |MoM Brent change| above threshold, broadcast to every corridor
# (matches how the Step 2 fundamentals sub-score itself is broadcast — Brent is a global driver,
# not corridor-specific, so a shock month is a candidate event for every corridor equally until
# you hand-assign which corridor(s) it actually maps to).
brent_mom = brent.pct_change()
shock_dates = brent_mom[brent_mom.abs() >= PRICE_SHOCK_MOM_THRESHOLD].index

price_shock_rows = []
for d in shock_dates:
    for c in CORRIDORS:
        price_shock_rows.append({
            "corridor_id": c, "candidate_date": d, "source": "price_shock",
            "trigger_value": float(brent_mom.loc[d]),
        })

# --- News-spike candidates: per-corridor news_subscore above its own high percentile,
# de-duplicated so a multi-day spike collapses to one representative candidate.
news_spike_rows = []
for c in CORRIDORS:
    sub = composite[composite["corridor_id"] == c].sort_values("as_of")
    if sub["news_subscore"].dropna().empty:
        continue
    cutoff = sub["news_subscore"].quantile(NEWS_SPIKE_PERCENTILE)
    spikes = sub[sub["news_subscore"] >= cutoff].copy()
    if spikes.empty:
        continue
    spikes["_group"] = (spikes["as_of"].diff() > timedelta(days=NEWS_SPIKE_DEDUP_DAYS)).cumsum()
    for _, grp in spikes.groupby("_group"):
        peak = grp.loc[grp["news_subscore"].idxmax()]
        news_spike_rows.append({
            "corridor_id": c, "candidate_date": peak["as_of"], "source": "news_spike",
            "trigger_value": float(peak["news_subscore"]),
        })

candidates = pd.DataFrame(price_shock_rows + news_spike_rows)
candidates = candidates.sort_values(["corridor_id", "candidate_date"]).reset_index(drop=True)
candidates.insert(0, "candidate_id", [f"bt{i:04d}" for i in range(len(candidates))])
candidates["is_genuine_event"] = ""      # fill: y / n
candidates["expected_severity"] = ""     # fill: 0.0-1.0
candidates["notes"] = ""                 # fill: what actually happened, if known

print(f"{len(candidates)} backtest candidates mined "
      f"({len(price_shock_rows)} price-shock rows across {len(shock_dates)} shock months, "
      f"{len(news_spike_rows)} news-spike rows)")
print(candidates.head(10))

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
candidates.to_csv(BACKTEST_CANDIDATES_OUTPUT, index=False)
print(f"\nWrote labeling template to {BACKTEST_CANDIDATES_OUTPUT} -- hand-fill "
      f"is_genuine_event / expected_severity / notes per plan Step 5.1 (aim for the ~30-50 "
      f"most plausible rows, not all of them).")

45 backtest candidates mined (24 price-shock rows across 4 shock months, 21 news-spike rows)
  candidate_id    corridor_id            candidate_date       source  trigger_value is_genuine_event expected_severity notes
0       bt0000  bab_el_mandeb 2025-07-08 06:00:00+00:00   news_spike       0.357096                                         
1       bt0001  bab_el_mandeb 2025-09-30 18:00:00+00:00   news_spike       0.360639                                         
2       bt0002  bab_el_mandeb 2026-01-31 00:00:00+00:00  price_shock       0.177669                                         
3       bt0003  bab_el_mandeb 2026-03-07 12:00:00+00:00   news_spike       0.324797                                         
4       bt0004  bab_el_mandeb 2026-03-26 18:00:00+00:00   news_spike       0.378756                                         
5       bt0005  bab_el_mandeb 2026-03-31 00:00:00+00:00  price_shock       0.776360                                         
6       bt0006  bab_el_mandeb 20

## 3. Lead-time metric (Step 5.3 / Step 6)

Only run against price-shock candidates — the Brent move itself is the objective event, so no
manual label is required to compute this one. For each shock, look back
`LEAD_TIME_LOOKBACK_DAYS` for the first date that corridor's composite score crossed
`COMPOSITE_TRIGGER_THRESHOLD_FOR_LEAD`. Positive `lead_days` means the composite led the price
move — the number plan Step 6 wants reported directly.

In [4]:
def first_crossing_before(corridor_df, shock_date, threshold, lookback_days):
    window_start = shock_date - timedelta(days=lookback_days)
    window = corridor_df[(corridor_df["as_of"] >= window_start) & (corridor_df["as_of"] <= shock_date)]
    hits = window[window["tier1_composite_score"] >= threshold]
    if hits.empty:
        return None
    return hits["as_of"].min()

lead_rows = []
for d in shock_dates:
    for c in CORRIDORS:
        corridor_df = composite[composite["corridor_id"] == c]
        crossing = first_crossing_before(corridor_df, d, COMPOSITE_TRIGGER_THRESHOLD_FOR_LEAD,
                                          LEAD_TIME_LOOKBACK_DAYS)
        lead_days = (d - crossing).total_seconds() / 86400 if crossing is not None else None
        lead_rows.append({
            "corridor_id": c, "shock_date": d, "brent_mom_change": float(brent_mom.loc[d]),
            "first_crossing": crossing, "lead_days": lead_days,
        })

lead_df = pd.DataFrame(lead_rows)
found = lead_df.dropna(subset=["lead_days"])

print(f"{len(found)} / {len(lead_df)} corridor-shock pairs had a composite crossing within "
      f"{LEAD_TIME_LOOKBACK_DAYS} days beforehand ({len(found) / max(len(lead_df), 1):.1%})")
if not found.empty:
    print(f"Lead time (days) -- mean: {found['lead_days'].mean():.1f}, "
          f"median: {found['lead_days'].median():.1f}, "
          f"min: {found['lead_days'].min():.1f}, max: {found['lead_days'].max():.1f}")
print(lead_df.sort_values(["shock_date", "corridor_id"]))

lead_df.to_csv(LEAD_TIME_OUTPUT, index=False)
print(f"\nWrote {LEAD_TIME_OUTPUT}")

2 / 24 corridor-shock pairs had a composite crossing within 30 days beforehand (8.3%)
Lead time (days) -- mean: 19.5, median: 19.5, min: 12.0, max: 27.0
      corridor_id                shock_date  brent_mom_change            first_crossing  lead_days
1   bab_el_mandeb 2026-01-31 00:00:00+00:00          0.177669                       NaT        NaN
0          hormuz 2026-01-31 00:00:00+00:00          0.177669                       NaT        NaN
5           india 2026-01-31 00:00:00+00:00          0.177669                       NaT        NaN
4    persian_gulf 2026-01-31 00:00:00+00:00          0.177669                       NaT        NaN
3         red_sea 2026-01-31 00:00:00+00:00          0.177669                       NaT        NaN
2            suez 2026-01-31 00:00:00+00:00          0.177669                       NaT        NaN
7   bab_el_mandeb 2026-03-31 00:00:00+00:00          0.776360                       NaT        NaN
6          hormuz 2026-03-31 00:00:00+00:00          0.

## 4. Trigger-threshold sweep (Step 5.4)

Reimplements Step 4's `compute_trigger` logic across a grid instead of one fixed pair, so you
can see the actual rate curve before picking new values for Step 4's config cell.

In [5]:
sub_cols = ["news_subscore", "vessel_subscore", "fundamentals_subscore"]
spread = composite[sub_cols].max(axis=1, skipna=True) - composite[sub_cols].min(axis=1, skipna=True)
n_subscores_present = composite[sub_cols].notna().sum(axis=1)
spread = spread.where(n_subscores_present > 1, 0.0)

sweep_rows = []
for ct in COMPOSITE_THRESHOLD_GRID:
    composite_hit = composite["tier1_composite_score"] >= ct
    for dt in DISAGREEMENT_THRESHOLD_GRID:
        disagreement_hit = spread >= dt
        triggered = composite_hit | disagreement_hit
        sweep_rows.append({
            "composite_threshold": ct, "disagreement_threshold": dt,
            "trigger_rate": float(triggered.mean()),
            "n_triggered": int(triggered.sum()),
            "n_composite_driven": int((composite_hit & ~disagreement_hit).sum()),
            "n_disagreement_driven": int((disagreement_hit & ~composite_hit).sum()),
            "n_both": int((composite_hit & disagreement_hit).sum()),
        })

sweep_df = pd.DataFrame(sweep_rows)
print(f"Current Step 4 config (composite={CURRENT_COMPOSITE_THRESHOLD}, "
      f"disagreement={CURRENT_DISAGREEMENT_THRESHOLD}) trigger rate:")
current_row = sweep_df[
    (sweep_df["composite_threshold"] == CURRENT_COMPOSITE_THRESHOLD)
    & (sweep_df["disagreement_threshold"] == CURRENT_DISAGREEMENT_THRESHOLD)
]
print(current_row.to_string(index=False) if not current_row.empty else "(not in grid -- add it to see directly)")

print("\nFull sweep:")
print(sweep_df.to_string(index=False))

sweep_df.to_csv(THRESHOLD_SWEEP_OUTPUT, index=False)
print(f"\nWrote {THRESHOLD_SWEEP_OUTPUT}")

Current Step 4 config (composite=0.65, disagreement=0.35) trigger rate:
 composite_threshold  disagreement_threshold  trigger_rate  n_triggered  n_composite_driven  n_disagreement_driven  n_both
                0.65                    0.35      0.330826         3035                   0                   3005      30

Full sweep:
 composite_threshold  disagreement_threshold  trigger_rate  n_triggered  n_composite_driven  n_disagreement_driven  n_both
                0.55                    0.30      0.374537         3436                   7                   2763     666
                0.55                    0.35      0.331589         3042                   7                   2369     666
                0.55                    0.40      0.277959         2550                   7                   1877     666
                0.55                    0.45      0.255287         2342                   7                   1669     666
                0.55                    0.50      0.24

## 5. Human-review disagreement threshold (Step 5.5)

Pulls the real `adjudicator.disagreement_index` distribution out of the Step 4 output and
suggests a percentile-based threshold instead of the flat 0.30 placeholder.

In [6]:
with open(STEP4_FINAL_RESULTS_PATH) as f:
    final_results = json.load(f)

disagreements = [
    r["adjudicator"]["disagreement_index"]
    for r in final_results.values()
    if r.get("adjudicator", {}).get("disagreement_index") is not None
]
disagreements = pd.Series(disagreements)
print(f"{len(disagreements)} episodes with a computable disagreement_index (of {len(final_results)} total)")

if not disagreements.empty:
    percentiles = [0.50, 0.75, 0.80, 0.90, 0.95]
    pct_table = pd.DataFrame({
        "percentile": percentiles,
        "disagreement_index_value": [disagreements.quantile(p) for p in percentiles],
    })
    print(pct_table.to_string(index=False))

    suggested_threshold = float(disagreements.quantile(HUMAN_REVIEW_PERCENTILE))
    n_flagged_current = int((disagreements > CURRENT_HUMAN_REVIEW_THRESHOLD).sum())
    n_flagged_suggested = int((disagreements > suggested_threshold).sum())
    print(f"\nCurrent threshold {CURRENT_HUMAN_REVIEW_THRESHOLD} flags {n_flagged_current} "
          f"episodes ({n_flagged_current / len(disagreements):.1%})")
    print(f"Suggested threshold ({HUMAN_REVIEW_PERCENTILE:.0%}ile = {suggested_threshold:.3f}) "
          f"flags {n_flagged_suggested} episodes ({n_flagged_suggested / len(disagreements):.1%}) "
          f"by construction")
else:
    suggested_threshold = None
    print("No disagreement_index values found -- was this a dry run (RUN_LIVE_CALLS=False) in Step 4?")

20 episodes with a computable disagreement_index (of 20 total)
 percentile  disagreement_index_value
       0.50                    0.1000
       0.75                    0.1625
       0.80                    0.2000
       0.90                    0.2000
       0.95                    0.2050

Current threshold 0.3 flags 0 episodes (0.0%)
Suggested threshold (80%ile = 0.200) flags 4 episodes (20.0%) by construction


## 6. Injection-resistance check — retrieval layer (Step 6, partial)

Injects one fabricated low-credibility "Hormuz closed" story into a **copy** of the real chunk
store (nothing is written back to your actual RAG files) and confirms it doesn't out-rank
genuine evidence for a matching query, thanks to the credibility-tier-sort from Step 3 Part 2.

This only exercises the retrieval layer. The other half of Step 6.4 — confirming Dove actually
argues the injected claim down rather than the composite score spiking — needs a live Hawk/Dove
call and isn't duplicated here; run it by dropping this fabricated row into your real corpus
and doing one `RUN_LIVE_CALLS=True` pass through the Step 4 notebook for that single episode.

In [7]:
chunks_df = pd.read_csv(RAG_CHUNKS_PATH)
chunks_df["as_of"] = pd.to_datetime(chunks_df["as_of"], utc=True)
rag_matrix = sparse.load_npz(RAG_MATRIX_PATH)
with open(RAG_VECTORIZER_PATH, "rb") as f:
    rag_vectorizer = pickle.load(f)
assert rag_matrix.shape[0] == len(chunks_df), "chunks_df / matrix row mismatch -- re-check Step 3 Part 1 export"

CREDIBILITY_ORDER = {"high": 0, "medium": 1, "low": 2}

def retrieve(chunks, matrix, vectorizer, corridor_id, as_of, query_text, top_k, lookback_days):
    as_of = pd.Timestamp(as_of)
    if as_of.tzinfo is None:
        as_of = as_of.tz_localize("UTC")
    window_start = as_of - timedelta(days=lookback_days)
    mask = (
        (chunks["corridor_id"] == corridor_id)
        & (chunks["as_of"] >= window_start)
        & (chunks["as_of"] <= as_of)
    )
    idx = np.flatnonzero(mask.to_numpy())
    if len(idx) == 0:
        return chunks.iloc[0:0].assign(similarity=pd.Series(dtype=float))
    sub_matrix = matrix[idx]
    from sklearn.metrics.pairwise import cosine_similarity
    sims = cosine_similarity(vectorizer.transform([query_text]), sub_matrix).flatten()
    sub = chunks.iloc[idx].copy()
    sub["similarity"] = sims
    sub["_cred_rank"] = sub["credibility_tier"].map(CREDIBILITY_ORDER).fillna(1)
    return sub.sort_values(["_cred_rank", "similarity"], ascending=[True, False]).head(top_k).drop(columns=["_cred_rank"])

# fabricate the injected row: most-recent as_of in the corridor, so it's inside any lookback window
recent_as_of = chunks_df.loc[chunks_df["corridor_id"] == INJECTION_TEST_CORRIDOR, "as_of"].max()
fake_text = "BREAKING strait of hormuz fully closed all tanker traffic halted immediately"
fake_row = {
    "chunk_id": "INJECTED_TEST:0", "story_id": "INJECTED_TEST", "chunk_index": 0,
    "chunk_text": fake_text, "corridor_id": INJECTION_TEST_CORRIDOR, "domain": "quicknews-africa.net",
    "credibility_tier": "low", "canonical_url": "//quicknews-africa.net/fake",
    "as_of": recent_as_of, "relevance_reason": "injected_test", "mention_count": 1,
    "text_source": "synthetic_injection_test",
}
test_chunks_df = pd.concat([chunks_df, pd.DataFrame([fake_row])], ignore_index=True)
fake_vec = rag_vectorizer.transform([fake_text])
test_matrix = sparse.vstack([rag_matrix, fake_vec])

hits = retrieve(test_chunks_df, test_matrix, rag_vectorizer, INJECTION_TEST_CORRIDOR, recent_as_of,
                 INJECTION_TEST_QUERY, INJECTION_TEST_TOP_K, INJECTION_TEST_LOOKBACK_DAYS)
print(hits[["chunk_id", "domain", "credibility_tier", "similarity"]].to_string(index=False))

injected_rank = hits.reset_index(drop=True).index[hits["chunk_id"] == "INJECTED_TEST:0"]
if len(injected_rank) == 0:
    print("\nPASS: injected low-credibility story did not make the top-k at all.")
elif injected_rank[0] < (hits["credibility_tier"] != "low").sum():
    print(f"\nCHECK: injected story ranked at position {injected_rank[0]} -- ahead of some "
          f"higher-credibility evidence. Investigate before trusting the tier-sort here.")
else:
    print(f"\nPASS: injected story appears (position {injected_rank[0]}) but ranked below all "
          f"higher-credibility evidence, as the credibility-tier-sort intends.")

          chunk_id        domain credibility_tier  similarity
ded4c0ee90732fbf:0  gcaptain.com             high    0.199106
d2449452d2ebc2a5:0      cnbc.com             high    0.186917
13c9b3c2d8adc19c:0      cnbc.com             high    0.165216
053fb9cb7295b005:0      cnbc.com             high    0.151164
fe051b6f8a9b2430:0  gcaptain.com             high    0.103962
6cf14a7a9fa4806e:0  gcaptain.com             high    0.097839
06a8187d85f45b63:0  gcaptain.com             high    0.094750
471f9844433820cb:0 aljazeera.com             high    0.083511

PASS: injected low-credibility story did not make the top-k at all.


## 7. Export calibrated config summary

In [8]:
calibrated_summary = {
    "generated_at": datetime.now(timezone.utc).isoformat(),
    "backtest_candidates_file": str(BACKTEST_CANDIDATES_OUTPUT),
    "n_backtest_candidates": int(len(candidates)),
    "lead_time": {
        "n_shock_corridor_pairs": int(len(lead_df)),
        "n_with_lead_signal": int(len(found)),
        "mean_lead_days": float(found["lead_days"].mean()) if not found.empty else None,
        "median_lead_days": float(found["lead_days"].median()) if not found.empty else None,
    },
    "trigger_threshold_sweep_file": str(THRESHOLD_SWEEP_OUTPUT),
    "current_trigger_config": {
        "composite_threshold": CURRENT_COMPOSITE_THRESHOLD,
        "disagreement_threshold": CURRENT_DISAGREEMENT_THRESHOLD,
    },
    "human_review_threshold": {
        "current": CURRENT_HUMAN_REVIEW_THRESHOLD,
        "suggested": suggested_threshold,
        "suggested_percentile": HUMAN_REVIEW_PERCENTILE,
    },
}

with open(CALIBRATED_CONFIG_OUTPUT, "w") as f:
    json.dump(calibrated_summary, f, indent=2, default=str)
print(f"Wrote {CALIBRATED_CONFIG_OUTPUT}")
print(json.dumps(calibrated_summary, indent=2, default=str))

Wrote outputs\step5_calibrated_config.json
{
  "generated_at": "2026-07-21T14:19:49.922681+00:00",
  "backtest_candidates_file": "outputs\\step5_backtest_candidates.csv",
  "n_backtest_candidates": 45,
  "lead_time": {
    "n_shock_corridor_pairs": 24,
    "n_with_lead_signal": 2,
    "mean_lead_days": 19.5,
    "median_lead_days": 19.5
  },
  "trigger_threshold_sweep_file": "outputs\\step5_threshold_sweep.csv",
  "current_trigger_config": {
    "composite_threshold": 0.65,
    "disagreement_threshold": 0.35
  },
  "human_review_threshold": {
    "current": 0.3,
    "suggested": 0.19999999999999996,
    "suggested_percentile": 0.8
  }
}


## What's still manual

- **Hand-label** `step5_backtest_candidates.csv` (`is_genuine_event`, `expected_severity`,
  `notes`) — this notebook mines candidates, it doesn't decide which are real.
- **Pick new thresholds** from Section 4's sweep table and edit them into Step 4's config cell
  (`COMPOSITE_TRIGGER_THRESHOLD`, `SUBSCORE_DISAGREEMENT_THRESHOLD`) — not auto-applied here.
- **Update `HUMAN_REVIEW_DISAGREEMENT_THRESHOLD`** in Step 4's config with Section 5's suggested
  value if you agree with it.
- **Run the live Dove-suppression half** of the injection test (Section 6) through Step 4 once,
  on the fabricated episode, to close out Step 6.4 fully.
- **Precision-on-filter spot-check** (Step 6, separate metric) — pull a random sample of Stage-B
  survivors from Step 1's corpus and manually confirm relevance; not covered here since it's a
  Step 1 filter question, not a Step 4/5 one.